In [0]:
%run ./00_utils

In [0]:
%pip install pandarallel

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

import warnings
import numpy as np
import pandas as pd
import mlflow
from pandarallel import pandarallel

pandarallel.initialize(nb_workers = 16, progress_bar=False)

In [0]:

# SCENARI WHAT IF
# scenario 1 (level 2)
# eredita rai 1
# eredita rai 2 group by (programma) 1.35 % accuracy | baseline 2.2%

# scenario 2 (level 1,5) -> ancora da sviluppare
# eredita rai 1 martedi
# eredita rai 1 giovedi groupby (programma, canale)

# SCENARIO PREDICTION PURA (level 1)
# eredita rai 1 8 sera martedi -> groupby (programma, canale, giorno della settimana, e ora)  1.0 % accuracy | baseline 1.1 %

# 1. Data Preparation

In [0]:
CUTOFF_DAYS = 7
CUTOFF_NUM_OCCURRENCES = 10
MIN_COMPETITOR_OVERLAP = 0.6
CATALOG = get_catalog()
TABLE_STORICO_PROGRAMMI = f"{CATALOG}.whatif.storico_programmi"

print('Reading df_programmi...')
df_programmi = read_df_programmi(table_name=TABLE_STORICO_PROGRAMMI)
                                 
print('Computing general historical stats...')
df_programmi = df_programmi.groupby('Programma').parallel_apply(
    compute_stats_last_occurrences, 
    suffix='', 
    cutoff_days=CUTOFF_DAYS, 
    cutoff_num_occurrences=CUTOFF_NUM_OCCURRENCES).reset_index(drop=True)

print('Computing precise historical stats...')
df_programmi = df_programmi.groupby(['Programma','Ora','Canale','GiornoSettimana']).parallel_apply(
    compute_stats_last_occurrences, 
    suffix='Precise', 
    cutoff_days=CUTOFF_DAYS, 
    cutoff_num_occurrences=CUTOFF_NUM_OCCURRENCES).reset_index(drop=True)

print('Computing previous and next program stats...')
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    df_programmi = df_programmi.groupby(['Canale']).parallel_apply(
        compute_stats_prev_next_program, 
        suffix='Precise').reset_index(drop=True)

df_programmi_pre = spark.createDataFrame(df_programmi)
df_programmi_pre = df_programmi_pre.where(F.col('StoricoShare') >= 0.0)

print('Done!')

In [0]:
df_programmi_pre = add_competitor(
    df_programmi_pre, 
    suffix='Precise', 
    min_competitor_overlap=MIN_COMPETITOR_OVERLAP)

In [0]:
df_programmi = df_programmi_pre

df_programmi = df_programmi.withColumn('ShareResiduo', F.col('Share') - F.col('StoricoShare'))
df_programmi = df_programmi.withColumn('HighValueShare', F.col('StoricoShare') >= 0.12)
df_programmi = df_programmi.withColumn('Programma_durata', F.col("ORA_FINE_TRX") - F.col("ORA_INIZIO_TRX"))
df_programmi = df_programmi.withColumn('StoricoShareLast_competitor', F.col('StoricoShareLast_competitor') - F.col('StoricoShare_competitor'))

for c in ['StoricoShareMax','StoricoShareMin','StoricoShareLast','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next']:
    df_programmi = df_programmi.withColumn(c, F.col(c) - F.col('StoricoShare'))

df_programmi = (
    df_programmi
    .withColumn('Ora', F.col('Ora').cast('string'))
    .withColumn('GiornoSettimana', F.col('GiornoSettimana').cast('string'))
    .withColumn('Mese', F.col('Mese').cast('string'))
)

df_programmi = (
    df_programmi.withColumn('StoricoShareUomini_delta', F.col('StoricoShareUomini') * F.col('StoricoShare') - F.col('StoricoShareUomini_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_08_14_delta', F.col('StoricoShare_08_14') * F.col('StoricoShare') - F.col('StoricoShare_08_14_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_15_24_delta', F.col('StoricoShare_15_24') * F.col('StoricoShare') - F.col('StoricoShare_15_24_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_25_64_delta', F.col('StoricoShare_25_64') * F.col('StoricoShare') - F.col('StoricoShare_25_64_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_65_plus_delta', F.col('StoricoShare_65_plus') * F.col('StoricoShare') - F.col('StoricoShare_65_plus_competitor') * F.col('StoricoShare_competitor'))
)

df_programmi = (
    df_programmi.where(F.col('Canale').isin(['Rai 1', 'Rai 2', 'Rai 3','Rete 4','Canale 5','Italia 1','Tv8']))
    .where(F.col('Ora') >= 7)
)

train_set = df_programmi.where(F.col('Data') <= '2026-03-31')
val_set = df_programmi.where((F.col('Data') > '2026-03-31') & (F.col('Data') <= '2026-04-30'))
test_set = df_programmi.where((F.col('Data') > '2026-04-30') & (F.col('Data') <= '2026-06-30'))

In [0]:
train_set.write.mode('overwrite').saveAsTable(f'{CATALOG}.whatif.train_set')
val_set.write.mode('overwrite').saveAsTable(f'{CATALOG}.whatif.val_set')
test_set.write.mode('overwrite').saveAsTable(f'{CATALOG}.whatif.test_set')

In [0]:
# main -        share: 10%, uomini: 40%
# competitor -  share: 12%, uomini: 70%
# competitor 2  share: 11%, uomini: 35% 

# delta vs competitor 1 -> 10*40 - 12*70 = - 0,044
# delta vs competitor 2 -> 10*40 - 11*35 = + 0.0015

In [0]:
train_set = spark.table(f'{CATALOG}.whatif.train_set')
val_set = spark.table(f'{CATALOG}.whatif.val_set')
test_set = spark.table(f'{CATALOG}.whatif.test_set')

In [0]:
test_set.where((F.col('Programma') == 'AFFARI TUOI') & (F.col("Data")=='2026-01-28')).display()

In [0]:
print(f'Train size: {train_set.count()}')
print(f'Val size: {val_set.count()}')
print(f'Test size: {test_set.count()}')

# 2. Model Training

## 2.1 Feature Definition

In [0]:
categorical_features = [
    'Canale',
    'DES_GENERE_ESTESA_INT',
    'DES_GENERE_FILM_INT',
    'DES_GENERE_SPORT_INT',
    'DES_MANIFESTAZIONE_SPORT_INT',
    # 'DES_SPECIALITA_SPORT_INT',
    'Canale_competitor',
    'DES_GENERE_ESTESA_INT_competitor',
    'FlgPrimaVisione',
    'FlgPrimaVisioneGen',
    'FlgPrimaVisioneSpec',
    # 'Cod_Genere_Int',
    'Ora',
    'GiornoSettimana',
    'Mese',
]

categorical_features_ohe = {}

for col in categorical_features:
    col_values = train_set.select(col).distinct().orderBy(col).collect()
    categorical_features_ohe[col] = []

    for value in col_values:
        if value[0] is None:
            continue
        value_name = value[0].replace(' ', '_').replace('.', '_').replace('-','_')
        train_set = train_set.withColumn(f'{col}_{value_name}', F.when(F.col(col) == value[0], 1).otherwise(0))
        val_set = val_set.withColumn(f'{col}_{value_name}', F.when(F.col(col) == value[0], 1).otherwise(0))
        test_set = test_set.withColumn(f'{col}_{value_name}', F.when(F.col(col) == value[0], 1).otherwise(0))

        categorical_features_ohe[col].append(f'{col}_{value_name}')

train_set = train_set.toPandas()
val_set = val_set.toPandas()
test_set = test_set.toPandas()

In [0]:
X_columns = [
    'ORA_INIZIO_TRX',
    'Programma_durata',
    # 'Ora',
    # 'GiornoSettimana',
    # 'Mese',
    # 'StoricoShare',
    'StoricoShareMax',
    'StoricoShareMin',
    'StoricoShareLast',
    # 'StoricoShareStd',
    'StoricoShare_competitor',
    'StoricoShareLast_competitor',
    # 'StoricoShareUomini',
    # 'StoricoShareUomini_competitor',
    'StoricoShareUomini_delta',
    # 'StoricoShare_08_14',
    # 'StoricoShare_08_14_competitor',
    'StoricoShare_08_14_delta',
    # 'StoricoShare_15_24',
    # 'StoricoShare_15_24_competitor',
    'StoricoShare_15_24_delta',
    # 'StoricoShare_25_64',
    # 'StoricoShare_25_64_competitor',
    'StoricoShare_25_64_delta',
    # 'StoricoShare_65_plus',
    # 'StoricoShare_65_plus_competitor',
    'StoricoShare_65_plus_delta',
    # 'overlap_perc_competitor',
    'StoricoShare_prev',
    'StoricoShare_next',
    'StoricoShareLast_prev',
    'StoricoShareLast_next',
    'StoricoShareUomini_prev',
    'StoricoShareUomini_next',
    'HighValueShare',
    # 'LowValueShare',
    # 'StoricoEtaMedia',
    # 'Share',
]

for cat_ohe, values in categorical_features_ohe.items():
    X_columns.extend(values)

y_column = 'ShareResiduo'

X_train, y_train = train_set[X_columns], train_set[y_column]
X_val, y_val = val_set[X_columns], val_set[y_column]
X_test, y_test = test_set[X_columns], test_set[y_column]

## 2.2 Training

In [0]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from itertools import product
from sklearn.metrics import mean_absolute_error
import mlflow

try:
    mlflow.end_run()
except:
    pass
mlflow.start_run()

params_grid = {
    'n_estimators': [2000],
    'early_stopping_rounds': [10],
    'learning_rate': [0.01],
    'max_depth': [12],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0.1],
}

keys = params_grid.keys()
values = params_grid.values()

best_params = None
best_eval = float('inf')
best_model = None

for combo in product(*values):
    params = dict(zip(keys, combo))

    model = XGBRegressor(
        **params,
        monotone_constraints = {
            'StoricoShare_competitor': -1, 'StoricoShareLast_competitor': -1, 
            'StoricoShareUomini_delta': -1,
            'StoricoShare_08_14_delta': -1,
            'StoricoShare_15_24_delta': -1,
            'StoricoShare_25_64_delta': -1,
            'StoricoShare_65_plus_delta': -1,
            # 'StoricoShare_prev': 1, 'StoricoShare_next': 1, 'StoricoShareLast_prev': 1, 'StoricoShareLast_next': 1, 'StoricoShareLast': 1,
        },
        random_state=42,
    )

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    y_val_pred = model.predict(X_val)

    eval = mean_absolute_error(y_val + val_set['StoricoShare'], y_val_pred + val_set['StoricoShare'])

    if eval < best_eval:
        best_eval = eval
        best_params = params
        best_model = model

    print(f'params: {params}, eval: {eval}')

model = best_model

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

mlflow.log_params(model.get_params())

mlflow.log_params({'X_columns': X_columns})
mlflow.log_params({'y_column': y_column})

In [0]:
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")

    mlflow.sklearn.log_model(model, 'model', input_example=X_test)

# 3. Output

## 3.1 Metrics

In [0]:
from sklearn.metrics import mean_absolute_error, r2_score

print('--- TRAIN SET ---')

_y_train = y_train + train_set['StoricoShare']
_y_train_pred = y_train_pred + train_set['StoricoShare']

mae = mean_absolute_error(_y_train, _y_train_pred)
print(f'MAE: {mae}')
r2 = r2_score(_y_train, _y_train_pred)
print(f'R2: {r2}')

mae_baseline = mean_absolute_error(_y_train, train_set['StoricoShare'])
print(f'MAE Baseline: {mae_baseline}')
r2_baseline = r2_score(_y_train, train_set['StoricoShare'])
print(f'R2 Baseline: {r2_baseline}')

mlflow.log_metrics({
    'mae_train': mae,
    'r2_train': r2,
    'mae_train_baseline': mae_baseline,
    'r2_train_baseline': r2_baseline,
})

print()
print('--- TEST SET ---')

_y_test = y_test + test_set['StoricoShare']
_y_test_pred = y_test_pred + test_set['StoricoShare']

mae = mean_absolute_error(_y_test, _y_test_pred)
print(f'MAE: {mae}')
r2 = r2_score(_y_test, _y_test_pred)
print(f'R2: {r2}')

mae_baseline = mean_absolute_error(_y_test, test_set['StoricoShare'])
print(f'MAE Baseline: {mae_baseline}')
r2_baseline = r2_score(_y_test, test_set['StoricoShare'])
print(f'R2 Baseline: {r2_baseline}')

mlflow.log_metrics({
    'mae_test': mae,
    'r2_test': r2,
    'mae_test_baseline': mae_baseline,
    'r2_test_baseline': r2_baseline,
})

mlflow.end_run()

In [0]:
test_set['SharePrevisto'] = _y_test_pred
test_set['ErroreAssoluto'] = abs(_y_test_pred - _y_test)

## 3.2 Confidence Intervals - XGBoost Quantiles

In [0]:
def fit_quantile_model(alpha):
    model_quantile = XGBRegressor(
        objective='reg:quantileerror',
        quantile_alpha=alpha,
        monotone_constraints=model.get_params()['monotone_constraints'],
        **best_params
    )
    model_quantile.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    return model_quantile

quantile_models = {
    '0.05': fit_quantile_model(0.05),
    '0.1': fit_quantile_model(0.1),
    '0.9': fit_quantile_model(0.9),
    '0.95': fit_quantile_model(0.95),
}

train_set['SharePrevisto05'] = quantile_models['0.05'].predict(X_train) + train_set['StoricoShare']
train_set['SharePrevisto10'] = quantile_models['0.1'].predict(X_train) + train_set['StoricoShare']
train_set['SharePrevisto90'] = quantile_models['0.9'].predict(X_train) + train_set['StoricoShare']
train_set['SharePrevisto95'] = quantile_models['0.95'].predict(X_train) + train_set['StoricoShare']

val_set['SharePrevisto05'] = quantile_models['0.05'].predict(X_val) + val_set['StoricoShare']
val_set['SharePrevisto10'] = quantile_models['0.1'].predict(X_val) + val_set['StoricoShare']
val_set['SharePrevisto90'] = quantile_models['0.9'].predict(X_val) + val_set['StoricoShare']
val_set['SharePrevisto95'] = quantile_models['0.95'].predict(X_val) + val_set['StoricoShare']

test_set['SharePrevisto05'] = quantile_models['0.05'].predict(X_test) + test_set['StoricoShare']
test_set['SharePrevisto10'] = quantile_models['0.1'].predict(X_test) + test_set['StoricoShare']
test_set['SharePrevisto90'] = quantile_models['0.9'].predict(X_test) + test_set['StoricoShare']
test_set['SharePrevisto95'] = quantile_models['0.95'].predict(X_test) + test_set['StoricoShare']

In [0]:
print('--- TRAIN SET ---')
print(f'Larg. media 10-90: ± {(train_set['SharePrevisto90'] - train_set['SharePrevisto10']).mean()/2*100:.2f} %')
print(f'Accuracy 10-90: {len(train_set.loc[(train_set['Share'] >= train_set['SharePrevisto10']) & (train_set['Share'] <= train_set['SharePrevisto90'])])/len(train_set)*100:.2f} %')
print()
print(f'Larg. media 05-95: ± {(train_set['SharePrevisto95'] - train_set['SharePrevisto05']).mean()/2*100:.2f} %')
print(f'Accuracy 05-95: {len(train_set.loc[(train_set['Share'] >= train_set['SharePrevisto05']) & (train_set['Share'] <= train_set['SharePrevisto95'])])/len(train_set)*100:.2f} %')
print()
print('--- VAL SET ---')
print(f'Larg. media 10-90: ± {(val_set['SharePrevisto90'] - val_set['SharePrevisto10']).mean()/2*100:.2f} %')
print(f'Accuracy 10-90: {len(val_set.loc[(val_set['Share'] >= val_set['SharePrevisto10']) & (val_set['Share'] <= val_set['SharePrevisto90'])])/len(val_set)*100:.2f} %')
print()
print(f'Larg. media 05-95: ± {(val_set['SharePrevisto95'] - val_set['SharePrevisto05']).mean()/2*100:.2f} %')
print(f'Accuracy 05-95: {len(val_set.loc[(val_set['Share'] >= val_set['SharePrevisto05']) & (val_set['Share'] <= val_set['SharePrevisto95'])])/len(val_set)*100:.2f} %')
print()
print('--- TEST SET ---')
print(f'Larg. media 10-90: ± {(test_set['SharePrevisto90'] - test_set['SharePrevisto10']).mean()/2*100:.2f} %')
print(f'Accuracy 10-90: {len(test_set.loc[(test_set['Share'] >= test_set['SharePrevisto10']) & (test_set['Share'] <= test_set['SharePrevisto90'])])/len(test_set)*100:.2f} %')
print()
print(f'Larg. media 05-95: ± {(test_set['SharePrevisto95'] - test_set['SharePrevisto05']).mean()/2*100:.2f} %')
print(f'Accuracy 05-95: {len(test_set.loc[(test_set['Share'] >= test_set['SharePrevisto05']) & (test_set['Share'] <= test_set['SharePrevisto95'])])/len(test_set)*100:.2f} %')

## 3.3 Graphs

In [0]:
display(spark.createDataFrame(test_set))

In [0]:
import pandas as pd

# df = pd.DataFrame(list(zip(_y_test_pred, _y_test, test_set['StoricoShare'])), columns=['Prediction', 'True', 'Storico'])

display(spark.createDataFrame(pd.DataFrame(list(zip(_y_test_pred, _y_test, test_set['StoricoShare'], test_set['ShareResiduo'], y_test, y_test + train_set['StoricoShare'])), columns=['Prediction', 'True', 'Storico','ShareResiduo', 'ShareResiduoLog', 'ShareLog'])))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
w = Window.orderBy("ORA_INIZIO_TRX")

df_step = (
    spark.createDataFrame(test_set)
    .where((F.col("Data") == "2026-01-28") & (F.col("Canale") == "Rai 1"))
    .withColumn("ORA_INIZIO_TRX", F.col("ORA_INIZIO_TRX") / 3600)
    .withColumn("ORA_FINE_TRX", F.col("ORA_FINE_TRX") / 3600)
    .withColumn("next_time", F.lead("ORA_INIZIO_TRX").over(w))
)

df_step = (
    df_step.select('Canale','Programma',"ORA_INIZIO_TRX", 'ORA_FINE_TRX', "Share", "SharePrevisto", 'StoricoShare', 'StoricoShare_prev','SharePrevisto05', 'SharePrevisto10', 'SharePrevisto90', 'SharePrevisto95')
    .unionByName(
        df_step.select(
            'Canale',
            'Programma',
            (F.col("next_time") - 0.0001).alias("ORA_INIZIO_TRX"),
            'ORA_FINE_TRX',
            "Share",
            "SharePrevisto",
            'StoricoShare',
            'StoricoShare_prev',
            'SharePrevisto05',
            'SharePrevisto10',
            'SharePrevisto90',
            'SharePrevisto95',
    ).where(F.col("next_time").isNotNull())
    ).withColumn('StoricoShare_prev', F.col('StoricoShare_prev') + F.col('StoricoShare'))
    .orderBy("ORA_INIZIO_TRX")
)

display(df_step)

Databricks visualization. Run in Databricks to view.

In [0]:
import matplotlib.pyplot as plt

_df = df_step.toPandas()

fig = plt.figure(figsize=(10,4), dpi=1000)

plt.grid()

plt.title(f'Share {_df['Canale'][0]} - Mercoledì 28 Gennaio 2026')

plt.plot(_df['ORA_INIZIO_TRX'], _df['Share']*100, label='ShareReale')
plt.plot(_df['ORA_INIZIO_TRX'], _df['SharePrevisto']*100, label='SharePrevisto')

plt.fill_between(_df['ORA_INIZIO_TRX'], _df['SharePrevisto05']*100, _df['SharePrevisto95']*100, alpha=0.2, label='IntervalloConfidenza')

plt.xlabel('Ora Trasmissione')
plt.ylabel('Share [%]')

plt.ylim(0,30)

plt.legend(loc='lower left')
plt.show()

In [0]:
feature_importance = pd.DataFrame(
    zip(model.feature_names_in_, model.feature_importances_),
    columns=['column', 'importance']
)

spark.createDataFrame(feature_importance).display()

In [0]:
display(spark.createDataFrame([(value,) for value in (_y_test_pred - _y_test)]).toDF('Errore'))

In [0]:
display(spark.createDataFrame(test_set).orderBy(F.col('ErroreAssoluto').desc()))

# 4. Inference

In [0]:
model_uri =f'models:/{CATALOG}.whatif.whatif_model_sostituzione/1'
model = mlflow.pyfunc.load_model(model_uri)
#model.predict(pd.DataFrame(X_test))

# 5. Model Interpretability

In [0]:
print('          train_p | train   | test')
print(f'E[f(X)] = {y_train_pred.mean():.5f} | {y_train.mean():.5f} | {y_test.mean():.5f} (residuo)')
print(f'E[f(X)] = {(y_train_pred + train_set['StoricoShare']).mean():.5f} | {(y_train + train_set['StoricoShare']).mean():.5f} | {(y_test + test_set['StoricoShare']).mean():.5f} (share)')

In [0]:
import shap

explainer = shap.TreeExplainer(model, model_output='raw')

# Load the raw sklearn model (XGBRegressor) for TreeExplainer.
# mlflow.pyfunc.load_model wraps the model and cannot be used directly with TreeExplainer, which requires the underlying booster for exact tree-traversal SHAP computation.
# raw_model = mlflow.sklearn.load_model(model_uri)
# explainer = shap.TreeExplainer(raw_model, model_output='raw')

In [0]:
from shap import Explanation

index = 7069
shap_values = explainer(X_test[index:index+1])

# from residual to original scale
shap_values = Explanation(
    values = shap_values[0].values * 100,
    base_values = (y_train_pred.mean() + test_set['StoricoShare'][index]) * 100,
    feature_names = X_test.columns,
    data = X_test.iloc[index],
)

# merge categorical features
shap_values_series = pd.Series(shap_values.values, index=X_test.columns)

new_shap_values = {}
new_data = {}

used_cols = set()

for f in categorical_features:
    
    prefix = f + '_'
    if 'competitor' in f:
        cols = [c for c in X_test.iloc[index].index if c.startswith(prefix)]
    else:
        cols = [c for c in X_test.iloc[index].index if c.startswith(prefix) and not c.startswith(prefix+'competitor_')]

    category_shap_value = shap_values_series[cols].sum()
    category_value = pd.to_numeric(X_test.iloc[index][cols], errors="coerce").idxmax().replace(prefix, '')
    
    new_shap_values[f] = category_shap_value
    new_data[f] = category_value

    used_cols.update(cols)

for col in X_test.iloc[index].index:
    if col not in used_cols:
        new_shap_values[col] = shap_values_series[col]
        new_data[col] = X_test.iloc[index][col]

# sum ORA_INIZIO_TRX + Ora
new_shap_values['Ora'] = new_shap_values['Ora'] + new_shap_values['ORA_INIZIO_TRX']
new_shap_values.pop('ORA_INIZIO_TRX')
new_data.pop('ORA_INIZIO_TRX')

# merge storico stats
new_shap_values['StoricoShare'] = new_shap_values['StoricoShareMax'] + new_shap_values['StoricoShareMin'] + new_shap_values['StoricoShareLast'] + new_shap_values['HighValueShare']
new_shap_values.pop('StoricoShareMax')
new_shap_values.pop('StoricoShareMin')
new_shap_values.pop('StoricoShareLast')
new_shap_values.pop('HighValueShare')

new_data['StoricoShare'] = 'Statistiche varie'
new_data.pop('StoricoShareMax')
new_data.pop('StoricoShareMin')
new_data.pop('StoricoShareLast')
new_data.pop('HighValueShare')

# merge storico stats competitor
new_shap_values['StoricoShare_competitor'] = new_shap_values['StoricoShareLast_competitor'] + new_shap_values['StoricoShare_competitor']
new_shap_values.pop('StoricoShareLast_competitor')

new_data['StoricoShare_competitor'] = 'Statistiche varie'
new_data.pop('StoricoShareLast_competitor')

# merge target age
new_shap_values['TargetEta_vs_competitor'] = new_shap_values['StoricoShare_08_14_delta'] + new_shap_values['StoricoShare_15_24_delta'] + new_shap_values['StoricoShare_25_64_delta'] + new_shap_values['StoricoShare_65_plus_delta']
new_shap_values.pop('StoricoShare_08_14_delta')
new_shap_values.pop('StoricoShare_15_24_delta')
new_shap_values.pop('StoricoShare_25_64_delta')
new_shap_values.pop('StoricoShare_65_plus_delta')

new_data['TargetEta_vs_competitor'] = 'Statistiche varie'
new_data.pop('StoricoShare_08_14_delta')
new_data.pop('StoricoShare_15_24_delta')
new_data.pop('StoricoShare_25_64_delta')
new_data.pop('StoricoShare_65_plus_delta')

# merge target gender
new_shap_values['TargetSesso_vs_competitor'] = new_shap_values['StoricoShareUomini_delta']
new_shap_values.pop('StoricoShareUomini_delta')

new_data['TargetSesso_vs_competitor'] = 'Statistiche varie'
new_data.pop('StoricoShareUomini_delta')

# merge programma precedente e successivo
new_shap_values['StoricoShare_precedente'] = new_shap_values['StoricoShare_prev'] + new_shap_values['StoricoShareLast_prev']
new_shap_values['StoricoShare_successivo'] = new_shap_values['StoricoShare_next'] + new_shap_values['StoricoShareLast_next']

new_shap_values.pop('StoricoShare_prev')
new_shap_values.pop('StoricoShareLast_prev')
new_shap_values.pop('StoricoShare_next')
new_shap_values.pop('StoricoShareLast_next')

new_data['StoricoShare_precedente'] = 'Statistiche varie'
new_data['StoricoShare_successivo'] = 'Statistiche varie'
new_data.pop('StoricoShare_prev')
new_data.pop('StoricoShareLast_prev')
new_data.pop('StoricoShare_next')
new_data.pop('StoricoShareLast_next')

storico_share = new_shap_values['StoricoShare']
new_shap_values.pop('StoricoShare')
new_data.pop('StoricoShare')

shap_values = Explanation(
    values = np.array(list(new_shap_values.values())),
    base_values = (y_train_pred.mean() + test_set['StoricoShare'][index] + storico_share/100) * 100,
    feature_names = np.array(list(new_shap_values.keys())),
    data = pd.Series(new_data),
)

print(f'programma      = {test_set['Programma'][index]}')
print(f'canale         = {test_set['Canale'][index]}')
print(f'data           = {test_set['Data'][index].strftime("%Y-%m-%d")}')
print(f'ora            = {test_set['Ora'][index]}')
print(f'competitor     = {test_set['Programma_competitor'][index]}')
print(f'canale comp.   = {new_data['Canale_competitor']}')
print(f'precedente     = {test_set['Programma_prev'][index]}')
print(f'storico prec.  = {test_set['StoricoShare_prev'][index] + test_set['StoricoShare'][index] * 100 :.2f} %')
print(f'successivo     = {test_set['Programma_next'][index]}')
print(f'storico succ.  = {test_set['StoricoShare_next'][index] + test_set['StoricoShare'][index] * 100 :.2f} %')
print('-'*40)
print(f'share storico  = {test_set['StoricoShare'][index] * 100:.2f} %')
print(f'share previsto = {test_set['SharePrevisto'][index] * 100:.2f} %')
print(f'share reale    = {test_set['Share'][index] * 100:.2f} %')

shap.waterfall_plot(shap_values, max_display=6)

In [0]:
from shap import Explanation

# ── 0. Selezione del record e calcolo SHAP sui valori raw ─────────────────────
index = 7069
shap_values = explainer(X_test[index:index+1])

# ── 1. Primo passaggio: cambio scala (da residual a share storico in percentuale) su tutte le features ─────────────────────
# Il modello calcola ShareResiduo, quindi gli SHAP hanno la stessa scala: moltiplichiamo *100 per portarli in percentuale.
# Spostiamo la baseline verso il valore di StoricoShare sommandocelo (necessario per usare gli SHAP per spiegare l'influenza su una predizione meramente sulla media dello ShareStorico): Base value = E[f(X_train)] + StoricoShare[index], che va ad ancorare la spiegazione al valore medio di ShareStorico.
shap_values = Explanation(
    values=shap_values[0].values * 100,
    base_values=(y_train_pred.mean() + test_set['StoricoShare'][index]) * 100,
    feature_names=X_test.columns,
    data=X_test.iloc[index],
)

shap_values_series = pd.Series(shap_values.values, index=X_test.columns)

new_shap_values = {}
new_data = {}
used_cols = set()

# ── 2. Aggregazione delle feature categoriche (che sono in forma OHE) ─────────────────────
# Sommiamo tutte le influenze di ogni versione dummy in una singola colonna.
# idxmax() serve a recuperare la colonna dummy "attiva" per questo record.
for f in categorical_features:
    prefix = f + '_'
    if 'competitor' in f:
        # Competitor
        cols = [c for c in X_test.iloc[index].index if c.startswith(prefix)]
    else:
        # Non-competitor (e.g. escludo 'Canale_competitor_*' quando aggrego con 'Canale')
        cols = [c for c in X_test.iloc[index].index
                if c.startswith(prefix) and not c.startswith(prefix + 'competitor_')]

    new_shap_values[f] = shap_values_series[cols].sum()
    new_data[f] = pd.to_numeric(X_test.iloc[index][cols], errors='coerce').idxmax().replace(prefix, '')
    used_cols.update(cols)

# Le feature non categoriche le ignoriamo
for col in X_test.iloc[index].index:
    if col not in used_cols:
        new_shap_values[col] = shap_values_series[col]
        new_data[col] = X_test.iloc[index][col]

# ── 3. Unione delle feature temporali  ─────────────────────
# Merge di ORA_INIZIO_TRX (secondi dalla mezzanotte) e Ora (ora categorica).
new_shap_values['Ora'] = new_shap_values['Ora'] + new_shap_values.pop('ORA_INIZIO_TRX')
new_data.pop('ORA_INIZIO_TRX')

# ── 4. Merge delle statistiche di ShareStorico dentro il base value ─────────────────────
#  Il loro SHAP aggregato verra' messo dentro il base value per non farle risaltare nel waterfall.
storico_cols = ['StoricoShareMax', 'StoricoShareMin', 'StoricoShareLast', 'HighValueShare']
new_shap_values['StoricoShare'] = sum(new_shap_values.pop(c) for c in storico_cols)
for c in storico_cols:
    new_data.pop(c)
new_data['StoricoShare'] = 'Statistiche varie'

# ── 5. Merge ShareStorico competitor ─────────────────────
new_shap_values['StoricoShare_competitor'] = (
    new_shap_values.pop('StoricoShareLast_competitor') + new_shap_values['StoricoShare_competitor']
)
new_data.pop('StoricoShareLast_competitor')
new_data['StoricoShare_competitor'] = 'Statistiche varie'

# ── 6. Merge delle feature sull'eta' ─────────────────────
# Somma dei valori riguardanti il delta sulle fasce d'eta' (08-14, 15-24, 25-64, 65+).
age_delta_cols = [
    'StoricoShare_08_14_delta', 'StoricoShare_15_24_delta',
    'StoricoShare_25_64_delta', 'StoricoShare_65_plus_delta',
]
new_shap_values['TargetEta_vs_competitor'] = sum(new_shap_values.pop(c) for c in age_delta_cols)
for c in age_delta_cols:
    new_data.pop(c)
new_data['TargetEta_vs_competitor'] = 'Statistiche varie'

# ── 7. Merge delle feature sul genere ─────────────────────
new_shap_values['TargetSesso_vs_competitor'] = new_shap_values.pop('StoricoShareUomini_delta')
new_data.pop('StoricoShareUomini_delta')
new_data['TargetSesso_vs_competitor'] = 'Statistiche varie'

# ── 8. Merge delle statistiche dei programmi precedenti e successivi  ─────────────────────
new_shap_values['StoricoShare_precedente'] = (
    new_shap_values.pop('StoricoShare_prev') + new_shap_values.pop('StoricoShareLast_prev')
)
new_shap_values['StoricoShare_successivo'] = (
    new_shap_values.pop('StoricoShare_next') + new_shap_values.pop('StoricoShareLast_next')
)
for c in ['StoricoShare_prev', 'StoricoShareLast_prev', 'StoricoShare_next', 'StoricoShareLast_next']:
    new_data.pop(c)
new_data['StoricoShare_precedente'] = 'Statistiche varie'
new_data['StoricoShare_successivo'] = 'Statistiche varie'

# ── 9. Secondo passaggio: assorbimento dello SHAP StoricoShare nella baseline (sulle features raggruppate) ─────────────────────
# Per nascondere i valori storici ed esaltare i valori piu' interessanti legati al contesto di quel programma.
storico_share = new_shap_values.pop('StoricoShare')
new_data.pop('StoricoShare')

shap_values = Explanation(
    values=np.array(list(new_shap_values.values())),
    base_values=(y_train_pred.mean() + test_set['StoricoShare'][index] + storico_share / 100) * 100,
    feature_names=np.array(list(new_shap_values.keys())),
    data=pd.Series(new_data),
)

# ── 10. Print ─────────────────────
print(f'programma      = {test_set["Programma"][index]}')
print(f'canale         = {test_set["Canale"][index]}')
print(f'data           = {test_set["Data"][index].strftime("%Y-%m-%d")}')
print(f'ora            = {test_set["Ora"][index]}')
print(f'competitor     = {test_set["Programma_competitor"][index]}')
print(f'canale comp.   = {new_data["Canale_competitor"]}')
print(f'precedente     = {test_set["Programma_prev"][index]}')
print(f'storico prec.  = {test_set["StoricoShare_prev"][index] + test_set["StoricoShare"][index] * 100:.2f} %')
print(f'successivo     = {test_set["Programma_next"][index]}')
print(f'storico succ.  = {test_set["StoricoShare_next"][index] + test_set["StoricoShare"][index] * 100:.2f} %')
print('-' * 40)
print(f'share storico  = {test_set["StoricoShare"][index] * 100:.2f} %')
print(f'share previsto = {test_set["SharePrevisto"][index] * 100:.2f} %')
print(f'share reale    = {test_set["Share"][index] * 100:.2f} %')

# ── 11. Waterfall  ─────────────────────
# Mostriamo le 6 feature piu' importanti per SHAP che spostano la previsione in su o in giu' rispetto al valore storico base
shap.waterfall_plot(shap_values, max_display=6)

In [0]:
from shap import Explanation
import shap
import numpy as np
import pandas as pd

def shap_waterfall_from_row(
    explainer,
    feature_row: pd.Series,
    storico_share: float,
    categorical_features: list,
    metadata: dict = None,
    max_display: int = 6,
):
    """
    Calcola e mostra il waterfall SHAP per un singolo record.

    Richiede SOLO:
      - explainer: shap.TreeExplainer costruito dal modello (contiene gia' expected_value)
      - feature_row: la riga di feature (304 colonne) usata dal modello per la predict
      - storico_share: lo share storico del programma (in scala 0-1), necessario per
                       riportare la spiegazione in scala percentuale assoluta
      - categorical_features: lista dei nomi delle feature categoriche originali
      - metadata: (opzionale) dict con campi descrittivi da stampare
                  es. {'Programma': '...', 'Canale': '...', 'Data': '...', ...}
    """
    # ── 0. Calcolo SHAP grezzo (dal modello, nessun dataset esterno) ───────────
    row_df = feature_row.to_frame().T if isinstance(feature_row, pd.Series) else feature_row
    raw_shap = explainer(row_df)

    # ── 1. Cambio scala: residuo -> percentuale di share ──────────────────────
    # explainer.expected_value == media delle predizioni del modello sul training (proprieta' intrinseca del TreeExplainer, non serve ricalcolarla)
    base_value_pct = (explainer.expected_value + storico_share) * 100

    shap_vals = Explanation(
        values=raw_shap[0].values * 100,
        base_values=base_value_pct,
        feature_names=row_df.columns,
        data=row_df.iloc[0],
    )

    shap_series = pd.Series(shap_vals.values, index=row_df.columns)
    new_shap_values = {}
    new_data = {}
    used_cols = set()

    # ── 2. Aggregazione feature categoriche (OHE -> singola) ──────────────────
    for f in categorical_features:
        prefix = f + '_'
        if 'competitor' in f:
            cols = [c for c in row_df.columns if c.startswith(prefix)]
        else:
            cols = [c for c in row_df.columns
                    if c.startswith(prefix) and not c.startswith(prefix + 'competitor_')]

        new_shap_values[f] = shap_series[cols].sum()
        new_data[f] = pd.to_numeric(row_df.iloc[0][cols], errors='coerce').idxmax().replace(prefix, '')
        used_cols.update(cols)

    for col in row_df.columns:
        if col not in used_cols:
            new_shap_values[col] = shap_series[col]
            new_data[col] = row_df.iloc[0][col]

    # ── 3. Merge feature temporali ────────────────────────────────────────────
    new_shap_values['Ora'] = new_shap_values['Ora'] + new_shap_values.pop('ORA_INIZIO_TRX')
    new_data.pop('ORA_INIZIO_TRX')

    # ── 4. Merge statistiche StoricoShare -> assorbite nella baseline ─────────
    storico_cols = ['StoricoShareMax', 'StoricoShareMin', 'StoricoShareLast', 'HighValueShare']
    new_shap_values['StoricoShare'] = sum(new_shap_values.pop(c) for c in storico_cols)
    for c in storico_cols:
        new_data.pop(c)
    new_data['StoricoShare'] = 'Statistiche varie'

    # ── 5. Merge StoricoShare competitor ──────────────────────────────────────
    new_shap_values['StoricoShare_competitor'] = (
        new_shap_values.pop('StoricoShareLast_competitor') + new_shap_values['StoricoShare_competitor']
    )
    new_data.pop('StoricoShareLast_competitor')
    new_data['StoricoShare_competitor'] = 'Statistiche varie'

    # ── 6. Merge feature eta' ─────────────────────────────────────────────────
    age_delta_cols = [
        'StoricoShare_08_14_delta', 'StoricoShare_15_24_delta',
        'StoricoShare_25_64_delta', 'StoricoShare_65_plus_delta',
    ]
    new_shap_values['TargetEta_vs_competitor'] = sum(new_shap_values.pop(c) for c in age_delta_cols)
    for c in age_delta_cols:
        new_data.pop(c)
    new_data['TargetEta_vs_competitor'] = 'Statistiche varie'

    # ── 7. Merge feature genere ───────────────────────────────────────────────
    new_shap_values['TargetSesso_vs_competitor'] = new_shap_values.pop('StoricoShareUomini_delta')
    new_data.pop('StoricoShareUomini_delta')
    new_data['TargetSesso_vs_competitor'] = 'Statistiche varie'

    # ── 8. Merge programmi precedente/successivo ──────────────────────────────
    new_shap_values['StoricoShare_precedente'] = (
        new_shap_values.pop('StoricoShare_prev') + new_shap_values.pop('StoricoShareLast_prev')
    )
    new_shap_values['StoricoShare_successivo'] = (
        new_shap_values.pop('StoricoShare_next') + new_shap_values.pop('StoricoShareLast_next')
    )
    for c in ['StoricoShare_prev', 'StoricoShareLast_prev', 'StoricoShare_next', 'StoricoShareLast_next']:
        new_data.pop(c, None)
    new_data['StoricoShare_precedente'] = 'Statistiche varie'
    new_data['StoricoShare_successivo'] = 'Statistiche varie'

    # ── 9. Assorbimento StoricoShare nella baseline ───────────────────────────
    storico_shap = new_shap_values.pop('StoricoShare')
    new_data.pop('StoricoShare')

    final_base = base_value_pct + storico_shap

    shap_explanation = Explanation(
        values=np.array(list(new_shap_values.values())),
        base_values=final_base,
        feature_names=np.array(list(new_shap_values.keys())),
        data=pd.Series(new_data),
    )

    # ── 10. Print metadata (se fornito) ───────────────────────────────────────
    if metadata:
        for k, v in metadata.items():
            print(f'{k:<15}= {v}')
        print('-' * 40)
    print(f'share storico  = {storico_share * 100:.2f} %')
    predicted = final_base + np.array(list(new_shap_values.values())).sum()
    print(f'share previsto = {predicted:.2f} %')

    # ── 11. Waterfall ─────────────────────────────────────────────────────────
    shap.waterfall_plot(shap_explanation, max_display=max_display)

    return shap_explanation


# ── Esempio di utilizzo (con i dati gia' in memoria) ─────────────────────────
index = 7069

_ = shap_waterfall_from_row(
    explainer=explainer,
    feature_row=X_test.iloc[index],
    storico_share=test_set['StoricoShare'][index],  # in serving: passato nel payload
    categorical_features=categorical_features,
    metadata={
        'Programma': test_set['Programma'][index],
        'Canale': test_set['Canale'][index],
        'Data': test_set['Data'][index].strftime('%Y-%m-%d'),
        'Ora': test_set['Ora'][index],
        'Competitor': test_set['Programma_competitor'][index],
    },
)

# 6. Simulazione

In [0]:
programma_main = test_set.loc[(test_set['Programma'] == 'PRIMA DI NOI') & (test_set['Data'] == '2026-01-11')].iloc[0]
# programma_main = test_set.loc[(test_set['Programma'] == 'CHI L\'HA VISTO ?') & (test_set['Data'] == '2026-01-28')].iloc[0]

programma_proposto = test_set.loc[(test_set['Programma'] =='REPORT') & (test_set['Canale'] == 'Rai 3') & (test_set['Ora'].isin(['20.0','21.0','22.0']))].iloc[-1]

print('-'*40)
print('          SIMULAZIONE WHAT IF')
print('-'*40)
print('--- PROGRAMMA INZIALE ---')
print(f'Canale         = {programma_main['Canale']}')
print(f'Programma      = {programma_main['Programma']}')
print(f'Ora inizio     = {round(float(programma_main['Ora']))}')
print(f'Ora inizio     = {programma_main['ORA_INIZIO_TRX'] / 3600}')
print(f'Share storico  = {programma_main['StoricoShare'] * 100:.2f} %')
print(f'Share previsto = {programma_main['SharePrevisto'] * 100:.2f} %')
max_value_index = np.argmax(programma_main[['StoricoShare_08_14', 'StoricoShare_15_24', 'StoricoShare_25_64', 'StoricoShare_65_plus']].values)
max_value = np.max(programma_main[['StoricoShare_08_14', 'StoricoShare_15_24', 'StoricoShare_25_64', 'StoricoShare_65_plus']].values)
max_category = None

if max_value_index == 0:
    max_category = '08-14'
elif max_value_index == 1:
    max_category = '15-24'
elif max_value_index == 2:
    max_category = '25-64'
elif max_value_index == 3:
    max_category = '65+'
print(f'Target eta     = {max_category} -> {max_value:.2f} %')
print(f'Target genere  = {'uomini' if programma_main['StoricoShareUomini'] >= 50 else 'donne'} -> {programma_main['StoricoShareUomini'] if programma_main['StoricoShareUomini'] >=50 else 100-programma_main['StoricoShareUomini']:.2f} %')
print()
print('--- COMPETITOR ---')
print(f'Canale         = {programma_main['Canale_competitor']}')
print(f'Programma      = {programma_main['Programma_competitor']}')
print(f'Share storico  = {programma_main['StoricoShare_competitor'] * 100:.2f} %')

max_value_index = np.argmax(programma_main[['StoricoShare_08_14_competitor', 'StoricoShare_15_24_competitor', 'StoricoShare_25_64_competitor', 'StoricoShare_65_plus_competitor']].values)
max_value = np.max(programma_main[['StoricoShare_08_14_competitor', 'StoricoShare_15_24_competitor', 'StoricoShare_25_64_competitor', 'StoricoShare_65_plus_competitor']].values)
max_category = None

if max_value_index == 0:
    max_category = '08-14'
elif max_value_index == 1:
    max_category = '15-24'
elif max_value_index == 2:
    max_category = '25-64'
elif max_value_index == 3:
    max_category = '65+'
print(f'Target eta     = {max_category} -> {max_value:.2f} %')
print(f'Target genere  = {'uomini' if programma_main['StoricoShareUomini_competitor'] >= 50 else 'donne'} -> {programma_main['StoricoShareUomini_competitor'] if programma_main['StoricoShareUomini_competitor'] >=50 else 100-programma_main['StoricoShareUomini_competitor']:.2f} %')

print()
print('--- PROGRAMMA PROPOSTO ---')
print(f'Programma      = {programma_proposto['Programma']}')
print(f'Ora inizio     = {round(float(programma_proposto['Ora']))}')
print(f'Share storico  = {programma_proposto['StoricoShare'] * 100:.2f} %')
max_value_index = np.argmax(programma_proposto[['StoricoShare_08_14', 'StoricoShare_15_24', 'StoricoShare_25_64', 'StoricoShare_65_plus']].values)
max_value = np.max(programma_proposto[['StoricoShare_08_14', 'StoricoShare_15_24', 'StoricoShare_25_64', 'StoricoShare_65_plus']].values)
max_category = None

if max_value_index == 0:
    max_category = '08-14'
elif max_value_index == 1:
    max_category = '15-24'
elif max_value_index == 2:
    max_category = '25-64'
elif max_value_index == 3:
    max_category = '65+'
print(f'Target eta     = {max_category} -> {max_value:.2f} %')
print(f'Target genere  = {'uomini' if programma_proposto['StoricoShareUomini'] >= 50 else 'donne'} -> {programma_proposto['StoricoShareUomini'] if programma_proposto['StoricoShareUomini'] >=50 else 100-programma_proposto['StoricoShareUomini']:.2f} %')

In [0]:
display(programma_proposto)


In [0]:
programma_proposto = test_set.loc[(test_set['Programma'] == 'PRIMA DI NOI') & (test_set['Data'] == '2026-01-11')].copy()

print(f'(1) NaN values: {programma_proposto.isna().sum().sum()}')

for c in ['StoricoShareMax','StoricoShareMin','StoricoShareLast','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next']:
    programma_proposto[c] = programma_proposto[c] + programma_proposto['StoricoShare']

print(f'(2) NaN values: {programma_proposto.isna().sum().sum()}')

columns_to_replace = ['Programma', 'Programma_durata','DES_GENERE_ESTESA_INT', 'DES_GENERE_FILM_INT', 'DES_GENERE_SPORT_INT', 'DES_MANIFESTAZIONE_SPORT_INT','StoricoShareMax', 'StoricoShareMin', 'StoricoShareLast','StoricoShare_08_14', 'StoricoShare_15_24', 'StoricoShare_25_64', 'StoricoShare_65_plus','StoricoShare', 'HighValueShare','StoricoShareUomini','DES_GENERE_ESTESA_INT_ANTEPRIMA_CINEMA_TV',
 "DES_GENERE_ESTESA_INT_ATTUALITA'",
 'DES_GENERE_ESTESA_INT_BALLO',
 'DES_GENERE_ESTESA_INT_CARTONI_ANIMATI',
 'DES_GENERE_ESTESA_INT_COMICHE',
 "DES_GENERE_ESTESA_INT_COSTUME_E_SOCIETA'",
 'DES_GENERE_ESTESA_INT_CUCINA',
 'DES_GENERE_ESTESA_INT_DIBATTITO_POLITICO',
 'DES_GENERE_ESTESA_INT_DOCUFICTION',
 'DES_GENERE_ESTESA_INT_DOCUMENTARIO',
 'DES_GENERE_ESTESA_INT_DOCUREALITY',
 'DES_GENERE_ESTESA_INT_FICTION_CICLO',
 'DES_GENERE_ESTESA_INT_FILM',
 'DES_GENERE_ESTESA_INT_FILM_CICLO',
 'DES_GENERE_ESTESA_INT_FILM_CORTOMETRAGGIO',
 'DES_GENERE_ESTESA_INT_FILM_DIBATTITO',
 'DES_GENERE_ESTESA_INT_FILM_TV',
 'DES_GENERE_ESTESA_INT_FOOD',
 'DES_GENERE_ESTESA_INT_GAME_SHOW',
 'DES_GENERE_ESTESA_INT_INCHIESTE',
 'DES_GENERE_ESTESA_INT_INFORMAZIONE_PARLAMENTARE',
 'DES_GENERE_ESTESA_INT_INTERRUZIONE',
 'DES_GENERE_ESTESA_INT_LIFESTYLE_ALTRO',
 'DES_GENERE_ESTESA_INT_LIRICA',
 'DES_GENERE_ESTESA_INT_MANIFESTAZIONI',
 'DES_GENERE_ESTESA_INT_MINISERIE',
 'DES_GENERE_ESTESA_INT_MONOGRAFIE',
 'DES_GENERE_ESTESA_INT_MUSICALE',
 'DES_GENERE_ESTESA_INT_PREVISIONI_DEL_TEMPO',
 'DES_GENERE_ESTESA_INT_PROGRAMMA_DI_MONTAGGIO',
 'DES_GENERE_ESTESA_INT_PROGRAMMA_PER_BAMBINI/RAGAZZI',
 'DES_GENERE_ESTESA_INT_PROGRAMMAZIONI_CINEMATOGRAFICHE',
 'DES_GENERE_ESTESA_INT_PROGRAMMI_DIBATTITO/INFORMATIVO',
 'DES_GENERE_ESTESA_INT_PROPERTY',
 'DES_GENERE_ESTESA_INT_PROSA',
 'DES_GENERE_ESTESA_INT_PROSSIMAMENTE',
 'DES_GENERE_ESTESA_INT_REALITY_SHOW',
 'DES_GENERE_ESTESA_INT_REDAZIONALE',
 'DES_GENERE_ESTESA_INT_ROTOCALCO_LEGGERO',
 'DES_GENERE_ESTESA_INT_RUBRICA',
 'DES_GENERE_ESTESA_INT_RUBRICA_RELIGIOSA',
 'DES_GENERE_ESTESA_INT_RUBRICA_SPORTIVA',
 'DES_GENERE_ESTESA_INT_RUBRICHE_DEL_TELEGIORNALE',
 'DES_GENERE_ESTESA_INT_RUBRICHE_DI_SERVIZIO',
 'DES_GENERE_ESTESA_INT_RUBRICHE_SCOLASTICHE',
 'DES_GENERE_ESTESA_INT_SANTA_MESSA',
 'DES_GENERE_ESTESA_INT_SCENEGGIATO',
 'DES_GENERE_ESTESA_INT_SCIENZA_ED_AMBIENTE',
 'DES_GENERE_ESTESA_INT_SEGNALE_ORARIO',
 'DES_GENERE_ESTESA_INT_SITUATION_COMEDY',
 'DES_GENERE_ESTESA_INT_SOAP_OPERA_/_TELEROMANZO',
 'DES_GENERE_ESTESA_INT_SPETTACOLO_MUSICALE',
 'DES_GENERE_ESTESA_INT_SPORT',
 'DES_GENERE_ESTESA_INT_STORIA/LETTERATURA/ARTE',
 'DES_GENERE_ESTESA_INT_TALENT_ALTRO',
 'DES_GENERE_ESTESA_INT_TALK_SHOW',
 'DES_GENERE_ESTESA_INT_TELEFILM',
 'DES_GENERE_ESTESA_INT_TELEGIORNALE',
 'DES_GENERE_ESTESA_INT_TELENOVELA',
 'DES_GENERE_ESTESA_INT_TG_SPORT',
 'DES_GENERE_ESTESA_INT_TRASMISSIONE_A_QUIZ',
 'DES_GENERE_ESTESA_INT_TRASMISSIONI_DI_SERVIZIO',
 'DES_GENERE_ESTESA_INT_TRIBUNA_POLITICA',
 'DES_GENERE_ESTESA_INT_TUTORIAL',
 "DES_GENERE_ESTESA_INT_VARIETA'",
 'DES_GENERE_ESTESA_INT_VIAGGI',
 'DES_GENERE_ESTESA_INT_WEDDING',
 'DES_GENERE_FILM_INT_ANIMAZIONE',
 'DES_GENERE_FILM_INT_AVVENTURA',
 'DES_GENERE_FILM_INT_AZIONE',
 'DES_GENERE_FILM_INT_BIOGRAFICO',
 'DES_GENERE_FILM_INT_COMMEDIA',
 'DES_GENERE_FILM_INT_COMMEDIA_MUSICALE',
 'DES_GENERE_FILM_INT_COMMEDIA_UMORISTICA',
 'DES_GENERE_FILM_INT_DOCUMENTARIO',
 'DES_GENERE_FILM_INT_DRAMMATICO',
 'DES_GENERE_FILM_INT_FANTASCIENZA',
 'DES_GENERE_FILM_INT_FANTASTICO',
 'DES_GENERE_FILM_INT_GENERE_CINEMAT__NON_ATTRIBUITO',
 'DES_GENERE_FILM_INT_GIALLO',
 'DES_GENERE_FILM_INT_GUERRA',
 'DES_GENERE_FILM_INT_MUSICALE',
 'DES_GENERE_FILM_INT_ORRORE',
 'DES_GENERE_FILM_INT_POLIZIESCO',
 'DES_GENERE_FILM_INT_SENTIMENTALE',
 'DES_GENERE_FILM_INT_SPIONAGGIO',
 'DES_GENERE_FILM_INT_STORICO',
 'DES_GENERE_FILM_INT_THRILLER',
 'DES_GENERE_FILM_INT_WESTERN',
 'DES_GENERE_SPORT_INT_ATLETICA_LEGGERA',
 'DES_GENERE_SPORT_INT_AUTOMOBILISMO',
 'DES_GENERE_SPORT_INT_CALCIO',
 'DES_GENERE_SPORT_INT_CANOA/CANOTTAGGIO',
 'DES_GENERE_SPORT_INT_CICLISMO',
 'DES_GENERE_SPORT_INT_FOOTBALL_AMERICANO',
 'DES_GENERE_SPORT_INT_GINNASTICA',
 'DES_GENERE_SPORT_INT_MOTOCICLISMO',
 'DES_GENERE_SPORT_INT_NUOTO',
 'DES_GENERE_SPORT_INT_Nessun_Genere',
 'DES_GENERE_SPORT_INT_PALLACANESTRO',
 'DES_GENERE_SPORT_INT_PALLANUOTO',
 'DES_GENERE_SPORT_INT_PALLAVOLO',
 'DES_GENERE_SPORT_INT_PATTINAGGIO_SU_GHIACCIO',
 'DES_GENERE_SPORT_INT_RUGBY',
 'DES_GENERE_SPORT_INT_SCHERMA',
 'DES_GENERE_SPORT_INT_SPORT_EQUESTRI',
 'DES_GENERE_SPORT_INT_SPORT_INVERNALI',
 'DES_GENERE_SPORT_INT_TENNIS',
 'DES_MANIFESTAZIONE_SPORT_INT_ALTRE_COPPE_MANIF_INTERNAZION_',
 'DES_MANIFESTAZIONE_SPORT_INT_ALTRE_COPPE_MANIF_ITALIANE',
 'DES_MANIFESTAZIONE_SPORT_INT_AVVENIMENTI_VARI',
 'DES_MANIFESTAZIONE_SPORT_INT_CAMPIONATO_DEL_MONDO',
 'DES_MANIFESTAZIONE_SPORT_INT_CAMPIONATO_EUROPEO',
 'DES_MANIFESTAZIONE_SPORT_INT_CAMPIONATO_ITALIANO',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_DEI_CAMPIONI',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_DEL_MONDO',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_INTERCONTINENTALE',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_ITALIA',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_UEFA',
 'DES_MANIFESTAZIONE_SPORT_INT_GIRI_REGIONALI/TROFEI_VARI',
 "DES_MANIFESTAZIONE_SPORT_INT_GIRO_D'ITALIA",
 'DES_MANIFESTAZIONE_SPORT_INT_GRAN_PREMIO_FORMULA_1',
 'DES_MANIFESTAZIONE_SPORT_INT_GRAN_PRIX_IAAF',
 "DES_MANIFESTAZIONE_SPORT_INT_INTERNAZIONALI_D'ITALIA",
 'DES_MANIFESTAZIONE_SPORT_INT_Nessun_Genere',
 'DES_MANIFESTAZIONE_SPORT_INT_PROVE_FORMULA_1',
 'DES_MANIFESTAZIONE_SPORT_INT_TORNEI_ATP',
 'DES_MANIFESTAZIONE_SPORT_INT_TORNEO_DELLE_6_NAZIONI',
 'DES_MANIFESTAZIONE_SPORT_INT_TOUR_DE_FRANCE',
 'DES_MANIFESTAZIONE_SPORT_INT_WIMBLEDON','FlgPrimaVisione_N',
 'FlgPrimaVisione_S',
 'FlgPrimaVisione_X',
 'FlgPrimaVisioneGen_9',
 'FlgPrimaVisioneGen_N',
 'FlgPrimaVisioneGen_S',
 'FlgPrimaVisioneSpec_9',
 'FlgPrimaVisioneSpec_N',
 'FlgPrimaVisioneSpec_S',]

columns_to_cast = ['DES_GENERE_ESTESA_INT_ANTEPRIMA_CINEMA_TV',
 "DES_GENERE_ESTESA_INT_ATTUALITA'",
 'DES_GENERE_ESTESA_INT_BALLO',
 'DES_GENERE_ESTESA_INT_CARTONI_ANIMATI',
 'DES_GENERE_ESTESA_INT_COMICHE',
 "DES_GENERE_ESTESA_INT_COSTUME_E_SOCIETA'",
 'DES_GENERE_ESTESA_INT_CUCINA',
 'DES_GENERE_ESTESA_INT_DIBATTITO_POLITICO',
 'DES_GENERE_ESTESA_INT_DOCUFICTION',
 'DES_GENERE_ESTESA_INT_DOCUMENTARIO',
 'DES_GENERE_ESTESA_INT_DOCUREALITY',
 'DES_GENERE_ESTESA_INT_FICTION_CICLO',
 'DES_GENERE_ESTESA_INT_FILM',
 'DES_GENERE_ESTESA_INT_FILM_CICLO',
 'DES_GENERE_ESTESA_INT_FILM_CORTOMETRAGGIO',
 'DES_GENERE_ESTESA_INT_FILM_DIBATTITO',
 'DES_GENERE_ESTESA_INT_FILM_TV',
 'DES_GENERE_ESTESA_INT_FOOD',
 'DES_GENERE_ESTESA_INT_GAME_SHOW',
 'DES_GENERE_ESTESA_INT_INCHIESTE',
 'DES_GENERE_ESTESA_INT_INFORMAZIONE_PARLAMENTARE',
 'DES_GENERE_ESTESA_INT_INTERRUZIONE',
 'DES_GENERE_ESTESA_INT_LIFESTYLE_ALTRO',
 'DES_GENERE_ESTESA_INT_LIRICA',
 'DES_GENERE_ESTESA_INT_MANIFESTAZIONI',
 'DES_GENERE_ESTESA_INT_MINISERIE',
 'DES_GENERE_ESTESA_INT_MONOGRAFIE',
 'DES_GENERE_ESTESA_INT_MUSICALE',
 'DES_GENERE_ESTESA_INT_PREVISIONI_DEL_TEMPO',
 'DES_GENERE_ESTESA_INT_PROGRAMMA_DI_MONTAGGIO',
 'DES_GENERE_ESTESA_INT_PROGRAMMA_PER_BAMBINI/RAGAZZI',
 'DES_GENERE_ESTESA_INT_PROGRAMMAZIONI_CINEMATOGRAFICHE',
 'DES_GENERE_ESTESA_INT_PROGRAMMI_DIBATTITO/INFORMATIVO',
 'DES_GENERE_ESTESA_INT_PROPERTY',
 'DES_GENERE_ESTESA_INT_PROSA',
 'DES_GENERE_ESTESA_INT_PROSSIMAMENTE',
 'DES_GENERE_ESTESA_INT_REALITY_SHOW',
 'DES_GENERE_ESTESA_INT_REDAZIONALE',
 'DES_GENERE_ESTESA_INT_ROTOCALCO_LEGGERO',
 'DES_GENERE_ESTESA_INT_RUBRICA',
 'DES_GENERE_ESTESA_INT_RUBRICA_RELIGIOSA',
 'DES_GENERE_ESTESA_INT_RUBRICA_SPORTIVA',
 'DES_GENERE_ESTESA_INT_RUBRICHE_DEL_TELEGIORNALE',
 'DES_GENERE_ESTESA_INT_RUBRICHE_DI_SERVIZIO',
 'DES_GENERE_ESTESA_INT_RUBRICHE_SCOLASTICHE',
 'DES_GENERE_ESTESA_INT_SANTA_MESSA',
 'DES_GENERE_ESTESA_INT_SCENEGGIATO',
 'DES_GENERE_ESTESA_INT_SCIENZA_ED_AMBIENTE',
 'DES_GENERE_ESTESA_INT_SEGNALE_ORARIO',
 'DES_GENERE_ESTESA_INT_SITUATION_COMEDY',
 'DES_GENERE_ESTESA_INT_SOAP_OPERA_/_TELEROMANZO',
 'DES_GENERE_ESTESA_INT_SPETTACOLO_MUSICALE',
 'DES_GENERE_ESTESA_INT_SPORT',
 'DES_GENERE_ESTESA_INT_STORIA/LETTERATURA/ARTE',
 'DES_GENERE_ESTESA_INT_TALENT_ALTRO',
 'DES_GENERE_ESTESA_INT_TALK_SHOW',
 'DES_GENERE_ESTESA_INT_TELEFILM',
 'DES_GENERE_ESTESA_INT_TELEGIORNALE',
 'DES_GENERE_ESTESA_INT_TELENOVELA',
 'DES_GENERE_ESTESA_INT_TG_SPORT',
 'DES_GENERE_ESTESA_INT_TRASMISSIONE_A_QUIZ',
 'DES_GENERE_ESTESA_INT_TRASMISSIONI_DI_SERVIZIO',
 'DES_GENERE_ESTESA_INT_TRIBUNA_POLITICA',
 'DES_GENERE_ESTESA_INT_TUTORIAL',
 "DES_GENERE_ESTESA_INT_VARIETA'",
 'DES_GENERE_ESTESA_INT_VIAGGI',
 'DES_GENERE_ESTESA_INT_WEDDING',
 'DES_GENERE_FILM_INT_ANIMAZIONE',
 'DES_GENERE_FILM_INT_AVVENTURA',
 'DES_GENERE_FILM_INT_AZIONE',
 'DES_GENERE_FILM_INT_BIOGRAFICO',
 'DES_GENERE_FILM_INT_COMMEDIA',
 'DES_GENERE_FILM_INT_COMMEDIA_MUSICALE',
 'DES_GENERE_FILM_INT_COMMEDIA_UMORISTICA',
 'DES_GENERE_FILM_INT_DOCUMENTARIO',
 'DES_GENERE_FILM_INT_DRAMMATICO',
 'DES_GENERE_FILM_INT_FANTASCIENZA',
 'DES_GENERE_FILM_INT_FANTASTICO',
 'DES_GENERE_FILM_INT_GENERE_CINEMAT__NON_ATTRIBUITO',
 'DES_GENERE_FILM_INT_GIALLO',
 'DES_GENERE_FILM_INT_GUERRA',
 'DES_GENERE_FILM_INT_MUSICALE',
 'DES_GENERE_FILM_INT_ORRORE',
 'DES_GENERE_FILM_INT_POLIZIESCO',
 'DES_GENERE_FILM_INT_SENTIMENTALE',
 'DES_GENERE_FILM_INT_SPIONAGGIO',
 'DES_GENERE_FILM_INT_STORICO',
 'DES_GENERE_FILM_INT_THRILLER',
 'DES_GENERE_FILM_INT_WESTERN',
 'DES_GENERE_SPORT_INT_ATLETICA_LEGGERA',
 'DES_GENERE_SPORT_INT_AUTOMOBILISMO',
 'DES_GENERE_SPORT_INT_CALCIO',
 'DES_GENERE_SPORT_INT_CANOA/CANOTTAGGIO',
 'DES_GENERE_SPORT_INT_CICLISMO',
 'DES_GENERE_SPORT_INT_FOOTBALL_AMERICANO',
 'DES_GENERE_SPORT_INT_GINNASTICA',
 'DES_GENERE_SPORT_INT_MOTOCICLISMO',
 'DES_GENERE_SPORT_INT_NUOTO',
 'DES_GENERE_SPORT_INT_Nessun_Genere',
 'DES_GENERE_SPORT_INT_PALLACANESTRO',
 'DES_GENERE_SPORT_INT_PALLANUOTO',
 'DES_GENERE_SPORT_INT_PALLAVOLO',
 'DES_GENERE_SPORT_INT_PATTINAGGIO_SU_GHIACCIO',
 'DES_GENERE_SPORT_INT_RUGBY',
 'DES_GENERE_SPORT_INT_SCHERMA',
 'DES_GENERE_SPORT_INT_SPORT_EQUESTRI',
 'DES_GENERE_SPORT_INT_SPORT_INVERNALI',
 'DES_GENERE_SPORT_INT_TENNIS',
 'DES_MANIFESTAZIONE_SPORT_INT_ALTRE_COPPE_MANIF_INTERNAZION_',
 'DES_MANIFESTAZIONE_SPORT_INT_ALTRE_COPPE_MANIF_ITALIANE',
 'DES_MANIFESTAZIONE_SPORT_INT_AVVENIMENTI_VARI',
 'DES_MANIFESTAZIONE_SPORT_INT_CAMPIONATO_DEL_MONDO',
 'DES_MANIFESTAZIONE_SPORT_INT_CAMPIONATO_EUROPEO',
 'DES_MANIFESTAZIONE_SPORT_INT_CAMPIONATO_ITALIANO',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_DEI_CAMPIONI',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_DEL_MONDO',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_INTERCONTINENTALE',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_ITALIA',
 'DES_MANIFESTAZIONE_SPORT_INT_COPPA_UEFA',
 'DES_MANIFESTAZIONE_SPORT_INT_GIRI_REGIONALI/TROFEI_VARI',
 "DES_MANIFESTAZIONE_SPORT_INT_GIRO_D'ITALIA",
 'DES_MANIFESTAZIONE_SPORT_INT_GRAN_PREMIO_FORMULA_1',
 'DES_MANIFESTAZIONE_SPORT_INT_GRAN_PRIX_IAAF',
 "DES_MANIFESTAZIONE_SPORT_INT_INTERNAZIONALI_D'ITALIA",
 'DES_MANIFESTAZIONE_SPORT_INT_Nessun_Genere',
 'DES_MANIFESTAZIONE_SPORT_INT_PROVE_FORMULA_1',
 'DES_MANIFESTAZIONE_SPORT_INT_TORNEI_ATP',
 'DES_MANIFESTAZIONE_SPORT_INT_TORNEO_DELLE_6_NAZIONI',
 'DES_MANIFESTAZIONE_SPORT_INT_TOUR_DE_FRANCE',
 'DES_MANIFESTAZIONE_SPORT_INT_WIMBLEDON','FlgPrimaVisione_N',
 'FlgPrimaVisione_S',
 'FlgPrimaVisione_X',
 'FlgPrimaVisioneGen_9',
 'FlgPrimaVisioneGen_N',
 'FlgPrimaVisioneGen_S',
 'FlgPrimaVisioneSpec_9',
 'FlgPrimaVisioneSpec_N',
 'FlgPrimaVisioneSpec_S']

programma_proposto[columns_to_replace] = pd.DataFrame(test_set.loc[(test_set['Programma'] =='REPORT') & (test_set['Canale'] == 'Rai 3') & (test_set['Data']=='2026-01-25') & (test_set['Ora'].isin(['20.0','21.0','22.0']))]).copy()[columns_to_replace].values

print(f'(3) NaN values: {programma_proposto.isna().sum().sum()}')

for c in ['StoricoShareMax','StoricoShareMin','StoricoShareLast','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next']:
    programma_proposto[c] = programma_proposto[c] - programma_proposto['StoricoShare']

print(f'(4) NaN values: {programma_proposto.isna().sum().sum()}')

programma_proposto['StoricoShareUomini_delta'] = programma_proposto['StoricoShareUomini'] * programma_proposto['StoricoShare'] - programma_proposto['StoricoShareUomini_competitor'] * programma_proposto['StoricoShare_competitor']
programma_proposto['StoricoShare_08_14_delta'] = programma_proposto['StoricoShare_08_14'] * programma_proposto['StoricoShare'] - programma_proposto['StoricoShare_08_14_competitor'] * programma_proposto['StoricoShare_competitor']
programma_proposto['StoricoShare_15_24_delta'] = programma_proposto['StoricoShare_15_24'] * programma_proposto['StoricoShare'] - programma_proposto['StoricoShare_15_24_competitor'] * programma_proposto['StoricoShare_competitor']
programma_proposto['StoricoShare_25_64_delta'] = programma_proposto['StoricoShare_25_64'] * programma_proposto['StoricoShare'] - programma_proposto['StoricoShare_25_64_competitor'] * programma_proposto['StoricoShare_competitor']
programma_proposto['StoricoShare_65_plus_delta'] = programma_proposto['StoricoShare_65_plus'] * programma_proposto['StoricoShare'] - programma_proposto['StoricoShare_65_plus_competitor'] * programma_proposto['StoricoShare_competitor']

print(f'(5) NaN values: {programma_proposto.isna().sum().sum()}')

# programma_proposto = pd.DataFrame(test_set.loc[(test_set['Programma'] =='REPORT') & (test_set['Canale'] == 'Rai 3') & (test_set['Data']=='2026-01-25') & (test_set['Ora'].isin(['20.0','21.0','22.0']))])

programma_proposto[columns_to_cast] = programma_proposto[columns_to_cast].astype(int)

programma_proposto[['StoricoShare_08_14_delta', 'StoricoShare_15_24_delta','StoricoShare_25_64_delta','StoricoShare_65_plus_delta','Programma_durata','StoricoShareMax','StoricoShareMin','StoricoShareLast','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next','HighValueShare','StoricoShareUomini_delta']] = programma_proposto[['StoricoShare_08_14_delta', 'StoricoShare_15_24_delta','StoricoShare_25_64_delta','StoricoShare_65_plus_delta','Programma_durata','StoricoShareMax','StoricoShareMin','StoricoShareLast','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next','HighValueShare','StoricoShareUomini_delta']].astype(float)

(model.predict(programma_proposto[X_columns]) + programma_proposto['StoricoShare']) * 100

In [0]:
X_columns

In [0]:
test_set.loc[(test_set['Programma'] =='REPORT') & (test_set['Canale'] == 'Rai 3') & (test_set['Data']=='2026-01-25') & (test_set['Ora'].isin(['20.0','21.0','22.0']))]

In [0]:
display(train_set.loc[
    (train_set['Canale'].isin(['Rai 1', 'Rai 2','Rai 3']))
    # & (train_set['StoricoShare_25_64'] < 65)
    # & (train_set['StoricoShareUomini'] >= 50)
    & (train_set['Ora'].isin(['19.0','20.0','21.0','22.0']))
    & (train_set['Programma_durata'] >= 60*60)
    & (train_set['StoricoShare'] >= 0.15)
][['Canale','Programma','GiornoSettimana']].drop_duplicates())

In [0]:
display(test_set.loc[
    # (np.argmax(test_set[['StoricoShare_08_14', 'StoricoShare_15_24', 'StoricoShare_25_64', 'StoricoShare_65_plus']].to_numpy(), axis=1) == np.argmax(test_set[['StoricoShare_08_14_competitor', 'StoricoShare_15_24_competitor', 'StoricoShare_25_64_competitor', 'StoricoShare_65_plus_competitor']].to_numpy(), axis=1))
    (((test_set['StoricoShareUomini'] >= 50) & (test_set['StoricoShareUomini_competitor'] >= 50)) | ((test_set['StoricoShareUomini'] < 50) & (test_set['StoricoShareUomini_competitor'] < 50)))
    & (test_set['Programma_durata'] >= 60*60)
    & (test_set['Canale_competitor'].isin(['Canale 5', 'Italia 1','Rete 4','Nove','La7','Tv8']))
    & (test_set['Canale'].isin(['Rai 1','Rai 2','Rai 3']))
    & (test_set['Ora'].isin(['20.0','21.0','22.0']))
][['Data','Ora', 'Programma', 'Canale', 'Share','SharePrevisto','Canale_competitor', 'Programma_competitor', 'StoricoShare_competitor','StoricoShareUomini','StoricoShareUomini_competitor']])